In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, roc_curve
from sklearn.metrics import precision_recall_curve, auc

from xgboost import XGBClassifier
import matplotlib.pyplot as plt

In [3]:
usage = pd.read_csv("../data/ravenstack_feature_usage.csv")
churn = pd.read_csv("../data/ravenstack_churn_events.csv")
subs = pd.read_csv("../data/ravenstack_subscriptions.csv")

In [ ]:
usage = usage.rename(columns={
    "usage_date": "timestamp",
    "usage_count": "activity"
})

churn = churn.rename(columns={
    "account_id": "user_id"
})

subs = subs.rename(columns={
    "account_id": "user_id"
})

In [ ]:
usage["timestamp"] = pd.to_datetime(usage["timestamp"])
churn["churn_date"] = pd.to_datetime(churn["churn_date"])
subs["start_date"] = pd.to_datetime(subs["start_date"])
subs["end_date"] = pd.to_datetime(subs["end_date"])

In [ ]:
usage = usage.merge(
    subs[["subscription_id", "user_id"]],
    on="subscription_id",
    how="left"
)

In [ ]:
feature_counts = usage.groupby(["user_id", "feature_name"]).size().unstack(fill_value=0)

feature_counts["num_features_used"] = (feature_counts > 0).sum(axis=1)
feature_counts["total_usage"] = feature_counts.sum(axis=1)

feature_counts = feature_counts.reset_index()

In [ ]:
usage = usage.groupby(["user_id", "timestamp"]).agg({
    "activity": "sum",
    "usage_duration_secs": "sum",
    "error_count": "sum"
}).reset_index()

In [ ]:
df = usage.merge(
    churn[[
        "user_id",
        "churn_date",
        "reason_code",
        "refund_amount_usd",
        "preceding_upgrade_flag",
        "preceding_downgrade_flag",
        "feedback_text"
    ]],
    on="user_id",
    how="left"
)

df = df.merge(
    subs[[
        "user_id",
        "plan_tier",
        "seats",
        "mrr_amount",
        "billing_frequency",
        "is_trial"
    ]],
    on="user_id",
    how="left"
)

In [ ]:
prediction_window = 30

df["churn"] = 0

df.loc[
    (df["churn_date"].notna()) &
    ((df["churn_date"] - df["timestamp"]).dt.days <= prediction_window) &
    ((df["churn_date"] - df["timestamp"]).dt.days >= 0),
    "churn"
] = 1

In [ ]:
def create_features(df):
    df = df.sort_values("timestamp")

    df["mean_7"] = df["activity"].rolling(7).mean()
    df["std_7"] = df["activity"].rolling(7).std()

    df["trend"] = df["activity"].diff()
    df["drop"] = df["activity"].pct_change()

    df["recent_activity"] = df["activity"].rolling(3).mean()
    df["activity_ratio"] = df["activity"] / (df["mean_7"] + 1)

    df["error_spike"] = (
        df["error_count"] > df["error_count"].rolling(7).mean()
    ).astype(int)

    df["days_since_last"] = df["timestamp"].diff().dt.days
    df["inactive"] = (df["days_since_last"] > 3).astype(int)

    df.replace([np.inf, -np.inf], 0, inplace=True)

    num_cols = df.select_dtypes(include=["float64", "int64"]).columns
    df[num_cols] = df[num_cols].fillna(0)

    return df

In [ ]:
df = df.sort_values(["user_id", "timestamp"])
df = df.groupby("user_id", group_keys=False).apply(create_features)
df = df.reset_index()

In [ ]:
df.columns
df = df.rename(columns={"index": "user_id"})
df.columns

In [ ]:
for col in ["preceding_upgrade_flag", "preceding_downgrade_flag", "is_trial"]:
    df[col] = df[col].astype(str).str.lower().map({
        "true": 1, "false": 0
    }).fillna(0)

df["plan_tier"] = df["plan_tier"].astype("category").cat.codes
df["billing_frequency"] = df["billing_frequency"].astype("category").cat.codes
df["reason_code"] = df["reason_code"].astype("category").cat.codes

df["has_feedback"] = df["feedback_text"].notna().astype(int)

In [ ]:
df["days_since_start"] = (df["timestamp"] - df.groupby("user_id")["timestamp"].transform("min")).dt.days

df["days_until_end"] = (df.groupby("user_id")["timestamp"].transform("max") - df["timestamp"]).dt.days

df["activity_decay"] = df["activity"] / (df["days_since_start"] + 1)

df["activity_trend_long"] = df["activity"].rolling(30).mean()

In [ ]:
df_user = df.groupby("user_id").agg({
    "activity": ["mean", "max", "std"],
    "usage_duration_secs": ["mean", "sum"],
    "error_count": ["mean", "sum", "max"],

    "mean_7": "mean",
    "std_7": "mean",
    "trend": "mean",
    "drop": "mean",

    "recent_activity": "mean",
    "activity_ratio": "mean",
    "error_spike": "mean",
    "inactive": "mean",
    "days_since_last": "mean",

    "seats": "max",
    "mrr_amount": "max",
    "plan_tier": "max",
    "billing_frequency": "max",
    "is_trial": "max",

    "reason_code": "max",
    "has_feedback": "max",

    "refund_amount_usd": "max",
    "preceding_upgrade_flag": "max",
    "preceding_downgrade_flag": "max",

    "churn": "max"
}).reset_index()

In [ ]:
df_user.columns = [
    "_".join(col).strip("_") for col in df_user.columns.values
]

In [ ]:
df_user["user_id"] = df_user["user_id"].astype(str)
feature_counts["user_id"] = feature_counts["user_id"].astype(str)

In [ ]:
df_user = df_user.merge(feature_counts, on="user_id", how="left")

In [ ]:
X = df_user.drop(columns=["user_id", "churn_max"])
y = df_user["churn_max"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [ ]:
scale = (y == 0).sum() / (y == 1).sum()

model = XGBClassifier(
    n_estimators=600,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale,
    eval_metric="aucpr",
    random_state=42
)

model.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import precision_recall_curve
y_probs = model.predict_proba(X_test)[:, 1]
precision, recall, thresholds = precision_recall_curve(y_test, y_probs)

f1 = 2 * (precision * recall) / (precision + recall + 1e-8)

best_idx = np.argmax(f1)
best_threshold = thresholds[best_idx]

print("Best threshold:", best_threshold)

In [ ]:
y_probs = model.predict_proba(X_test)[:, 1]
y_pred = (y_probs > best_threshold).astype(int)

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_probs))

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_probs)

plt.plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_test, y_probs):.3f}")
plt.plot([0,1],[0,1],'--')
plt.legend()
plt.show()

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_test, y_probs)

pr_auc = auc(recall, precision)

print("PR-AUC:", pr_auc)

In [ ]:
plt.plot(recall, precision, label=f"PR-AUC = {pr_auc:.3f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.legend()
plt.show()

In [ ]:
plt.plot(thresholds, f1[:-1])
plt.xlabel("Threshold")
plt.ylabel("F1 Score")
plt.show()